In [1]:
%%capture
!pip install hierarchicalforecast statsforecast

In [2]:
import numpy as np
import pandas as pd


In [3]:
Y_df = pd.read_csv('https://raw.githubusercontent.com/Nixtla/transfer-learning-time-series/main/datasets/tourism.csv')


In [4]:
Y_df

,Quarter,Region,State,Purpose,Trips
0,1998 Q1,Adelaide,South Australia,Business,135.077690
1,1998 Q2,Adelaide,South Australia,Business,109.987316
2,1998 Q3,Adelaide,South Australia,Business,166.034687
3,1998 Q4,Adelaide,South Australia,Business,127.160464
4,1999 Q1,Adelaide,South Australia,Business,137.448533
...,...,...,...,...,...
24315,2016 Q4,Yorke Peninsula,South Australia,Visiting,33.672151
24316,2017 Q1,Yorke Peninsula,South Australia,Visiting,46.223014
24317,2017 Q2,Yorke Peninsula,South Australia,Visiting,50.582837
24318,2017 Q3,Yorke Peninsula,South Australia,Visiting,27.766728


In [5]:
Y_df = Y_df.rename({'Trips': 'y', 'Quarter': 'ds'}, axis=1)
Y_df

,ds,Region,State,Purpose,y
0,1998 Q1,Adelaide,South Australia,Business,135.077690
1,1998 Q2,Adelaide,South Australia,Business,109.987316
2,1998 Q3,Adelaide,South Australia,Business,166.034687
3,1998 Q4,Adelaide,South Australia,Business,127.160464
4,1999 Q1,Adelaide,South Australia,Business,137.448533
...,...,...,...,...,...
24315,2016 Q4,Yorke Peninsula,South Australia,Visiting,33.672151
24316,2017 Q1,Yorke Peninsula,South Australia,Visiting,46.223014
24317,2017 Q2,Yorke Peninsula,South Australia,Visiting,50.582837
24318,2017 Q3,Yorke Peninsula,South Australia,Visiting,27.766728


In [6]:
Y_df.insert(0, 'Country', 'Australia')
Y_df

,Country,ds,Region,State,Purpose,y
0,Australia,1998 Q1,Adelaide,South Australia,Business,135.077690
1,Australia,1998 Q2,Adelaide,South Australia,Business,109.987316
2,Australia,1998 Q3,Adelaide,South Australia,Business,166.034687
3,Australia,1998 Q4,Adelaide,South Australia,Business,127.160464
4,Australia,1999 Q1,Adelaide,South Australia,Business,137.448533
...,...,...,...,...,...,...
24315,Australia,2016 Q4,Yorke Peninsula,South Australia,Visiting,33.672151
24316,Australia,2017 Q1,Yorke Peninsula,South Australia,Visiting,46.223014
24317,Australia,2017 Q2,Yorke Peninsula,South Australia,Visiting,50.582837
24318,Australia,2017 Q3,Yorke Peninsula,South Australia,Visiting,27.766728


In [7]:
Y_df = Y_df[['Country', 'Region', 'State', 'Purpose', 'ds', 'y']]

Y_df

,Country,Region,State,Purpose,ds,y
0,Australia,Adelaide,South Australia,Business,1998 Q1,135.077690
1,Australia,Adelaide,South Australia,Business,1998 Q2,109.987316
2,Australia,Adelaide,South Australia,Business,1998 Q3,166.034687
3,Australia,Adelaide,South Australia,Business,1998 Q4,127.160464
4,Australia,Adelaide,South Australia,Business,1999 Q1,137.448533
...,...,...,...,...,...,...
24315,Australia,Yorke Peninsula,South Australia,Visiting,2016 Q4,33.672151
24316,Australia,Yorke Peninsula,South Australia,Visiting,2017 Q1,46.223014
24317,Australia,Yorke Peninsula,South Australia,Visiting,2017 Q2,50.582837
24318,Australia,Yorke Peninsula,South Australia,Visiting,2017 Q3,27.766728


In [8]:
Y_df['ds'] = Y_df['ds'].str.replace(r'(\d+) (Q\d)', r'\1-\2', regex=True)
Y_df

,Country,Region,State,Purpose,ds,y
0,Australia,Adelaide,South Australia,Business,1998-Q1,135.077690
1,Australia,Adelaide,South Australia,Business,1998-Q2,109.987316
2,Australia,Adelaide,South Australia,Business,1998-Q3,166.034687
3,Australia,Adelaide,South Australia,Business,1998-Q4,127.160464
4,Australia,Adelaide,South Australia,Business,1999-Q1,137.448533
...,...,...,...,...,...,...
24315,Australia,Yorke Peninsula,South Australia,Visiting,2016-Q4,33.672151
24316,Australia,Yorke Peninsula,South Australia,Visiting,2017-Q1,46.223014
24317,Australia,Yorke Peninsula,South Australia,Visiting,2017-Q2,50.582837
24318,Australia,Yorke Peninsula,South Australia,Visiting,2017-Q3,27.766728


In [9]:
Y_df['ds'] = pd.PeriodIndex(Y_df["ds"], freq='Q').to_timestamp()
Y_df

,Country,Region,State,Purpose,ds,y
0,Australia,Adelaide,South Australia,Business,1998-01-01,135.077690
1,Australia,Adelaide,South Australia,Business,1998-04-01,109.987316
2,Australia,Adelaide,South Australia,Business,1998-07-01,166.034687
3,Australia,Adelaide,South Australia,Business,1998-10-01,127.160464
4,Australia,Adelaide,South Australia,Business,1999-01-01,137.448533
...,...,...,...,...,...,...
24315,Australia,Yorke Peninsula,South Australia,Visiting,2016-10-01,33.672151
24316,Australia,Yorke Peninsula,South Australia,Visiting,2017-01-01,46.223014
24317,Australia,Yorke Peninsula,South Australia,Visiting,2017-04-01,50.582837
24318,Australia,Yorke Peninsula,South Australia,Visiting,2017-07-01,27.766728


In [10]:
spec = [
    ['Country'],
    ['Country', 'State'], 
    ['Country', 'Purpose'], 
    ['Country', 'State', 'Region'], 
    ['Country', 'State', 'Purpose'], 
    ['Country', 'State', 'Region', 'Purpose']
]

In [11]:
from hierarchicalforecast.utils import aggregate


In [12]:
%%capture
Y_df, S_df, tags = aggregate(Y_df, spec)

In [13]:
Y_df

,unique_id,ds,y
0,Australia,1998-01-01,23182.197269
1,Australia,1998-04-01,20323.380067
2,Australia,1998-07-01,19826.640511
3,Australia,1998-10-01,20830.129891
4,Australia,1999-01-01,22087.353380
...,...,...,...
33995,Australia/Western Australia/Experience Perth/V...,2016-10-01,439.699451
33996,Australia/Western Australia/Experience Perth/V...,2017-01-01,356.867038
33997,Australia/Western Australia/Experience Perth/V...,2017-04-01,302.296119
33998,Australia/Western Australia/Experience Perth/V...,2017-07-01,373.442070


In [14]:
S_df

,unique_id,Australia/ACT/Canberra/Business,Australia/ACT/Canberra/Holiday,Australia/ACT/Canberra/Other,Australia/ACT/Canberra/Visiting,Australia/New South Wales/Blue Mountains/Business,Australia/New South Wales/Blue Mountains/Holiday,Australia/New South Wales/Blue Mountains/Other,Australia/New South Wales/Blue Mountains/Visiting,Australia/New South Wales/Capital Country/Business,...,Australia/Western Australia/Australia's North West/Other,Australia/Western Australia/Australia's North West/Visiting,Australia/Western Australia/Australia's South West/Business,Australia/Western Australia/Australia's South West/Holiday,Australia/Western Australia/Australia's South West/Other,Australia/Western Australia/Australia's South West/Visiting,Australia/Western Australia/Experience Perth/Business,Australia/Western Australia/Experience Perth/Holiday,Australia/Western Australia/Experience Perth/Other,Australia/Western Australia/Experience Perth/Visiting
0,Australia,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
1,Australia/ACT,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,Australia/New South Wales,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,Australia/Northern Territory,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,Australia/Queensland,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
420,Australia/Western Australia/Australia's South ...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
421,Australia/Western Australia/Experience Perth/B...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
422,Australia/Western Australia/Experience Perth/H...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
423,Australia/Western Australia/Experience Perth/O...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [16]:
tags

{'Country': array(['Australia'], dtype=object),
 'Country/State': array(['Australia/ACT', 'Australia/New South Wales',
        'Australia/Northern Territory', 'Australia/Queensland',
        'Australia/South Australia', 'Australia/Tasmania',
        'Australia/Victoria', 'Australia/Western Australia'], dtype=object),
 'Country/Purpose': array(['Australia/Business', 'Australia/Holiday', 'Australia/Other',
        'Australia/Visiting'], dtype=object),
 'Country/State/Region': array(['Australia/ACT/Canberra',
        'Australia/New South Wales/Blue Mountains',
        'Australia/New South Wales/Capital Country',
        'Australia/New South Wales/Central Coast',
        'Australia/New South Wales/Central NSW',
        'Australia/New South Wales/Hunter',
        'Australia/New South Wales/New England North West',
        'Australia/New South Wales/North Coast NSW',
        'Australia/New South Wales/Outback NSW',
        'Australia/New South Wales/Riverina',
        'Australia/New South Wa

In [17]:
Y_test_df = Y_df.groupby('unique_id', as_index=False).tail(8)
Y_train_df = Y_df.drop(Y_test_df.index)

In [18]:
Y_train_df.groupby('unique_id').size()

unique_id
Australia                                                72
Australia/ACT                                            72
Australia/ACT/Business                                   72
Australia/ACT/Canberra                                   72
Australia/ACT/Canberra/Business                          72
                                                         ..
Australia/Western Australia/Experience Perth/Other       72
Australia/Western Australia/Experience Perth/Visiting    72
Australia/Western Australia/Holiday                      72
Australia/Western Australia/Other                        72
Australia/Western Australia/Visiting                     72
Length: 425, dtype: int64

In [19]:
Y_test_df.groupby('unique_id').size()

unique_id
Australia                                                8
Australia/ACT                                            8
Australia/ACT/Business                                   8
Australia/ACT/Canberra                                   8
Australia/ACT/Canberra/Business                          8
                                                        ..
Australia/Western Australia/Experience Perth/Other       8
Australia/Western Australia/Experience Perth/Visiting    8
Australia/Western Australia/Holiday                      8
Australia/Western Australia/Other                        8
Australia/Western Australia/Visiting                     8
Length: 425, dtype: int64

In [20]:
%%capture
from statsforecast.models import AutoETS
from statsforecast.core import StatsForecast

In [21]:
fcst = StatsForecast(models=[AutoETS(season_length=4, model='ZZA')], 
                     freq='QS', n_jobs=-1)
Y_hat_df = fcst.forecast(df=Y_train_df, h=8, fitted=True)
Y_fitted_df = fcst.forecast_fitted_values()

In [23]:
len(Y_hat_df) , len(Y_fitted_df) , len(Y_train_df) , len(Y_test_df)

(3400, 30600, 30600, 3400)

In [25]:
Y_fitted_df.head()

,unique_id,ds,y,AutoETS
0,Australia,1998-01-01,23182.197269,22521.177714
1,Australia,1998-04-01,20323.380067,21340.083713
2,Australia,1998-07-01,19826.640511,20316.741658
3,Australia,1998-10-01,20830.129891,20646.219131
4,Australia,1999-01-01,22087.353380,22170.448469


In [26]:
from hierarchicalforecast.methods import BottomUp, MinTrace
from hierarchicalforecast.core import HierarchicalReconciliation

In [27]:
reconcilers = [
    BottomUp(),
    MinTrace(method='mint_shrink'),
    MinTrace(method='ols')
]
hrec = HierarchicalReconciliation(reconcilers=reconcilers)
Y_rec_df = hrec.reconcile(Y_hat_df=Y_hat_df, Y_df=Y_fitted_df, S_df=S_df, tags=tags)

In [33]:
Y_rec_df.head() , Y_hat_df.head()

(   unique_id         ds       AutoETS  AutoETS/BottomUp  \
 0  Australia 2016-01-01  25990.068004      24381.672903   
 1  Australia 2016-04-01  24458.490282      22903.194015   
 2  Australia 2016-07-01  23974.055984      22411.401316   
 3  Australia 2016-10-01  24563.454495      23127.009693   
 4  Australia 2017-01-01  25990.068004      24518.047370   
 
    AutoETS/MinTrace_method-mint_shrink  AutoETS/MinTrace_method-ols  
 0                         25427.793552                 25894.419896  
 1                         23913.800914                 24357.231461  
 2                         23428.540858                 23865.928094  
 3                         24089.585592                 24470.780870  
 4                         25545.039186                 25901.383310  ,
    unique_id         ds       AutoETS
 0  Australia 2016-01-01  25990.068004
 1  Australia 2016-04-01  24458.490282
 2  Australia 2016-07-01  23974.055984
 3  Australia 2016-10-01  24563.454495
 4  Australia 20

In [34]:
from hierarchicalforecast.evaluation import evaluate
from utilsforecast.losses import rmse, mase
from functools import partial

In [35]:
eval_tags = {}
eval_tags['Total'] = tags['Country']
eval_tags['Purpose'] = tags['Country/Purpose']
eval_tags['State'] = tags['Country/State']
eval_tags['Regions'] = tags['Country/State/Region']
eval_tags['Bottom'] = tags['Country/State/Region/Purpose']

In [37]:
Y_test_df.head()

,unique_id,ds,y
72,Australia,2016-01-01,26660.637689
73,Australia,2016-04-01,24285.027757
74,Australia,2016-07-01,24191.320131
75,Australia,2016-10-01,26347.600973
76,Australia,2017-01-01,27496.389021


In [39]:
Y_rec_df.head()

,unique_id,ds,AutoETS,AutoETS/BottomUp,AutoETS/MinTrace_method-mint_shrink,AutoETS/MinTrace_method-ols
0,Australia,2016-01-01,25990.068004,24381.672903,25427.793552,25894.419896
1,Australia,2016-04-01,24458.490282,22903.194015,23913.800914,24357.231461
2,Australia,2016-07-01,23974.055984,22411.401316,23428.540858,23865.928094
3,Australia,2016-10-01,24563.454495,23127.009693,24089.585592,24470.780870
4,Australia,2017-01-01,25990.068004,24518.047370,25545.039186,25901.383310


In [40]:
df = Y_rec_df.merge(Y_test_df, on=['unique_id', 'ds'])
df.head()

,unique_id,ds,AutoETS,AutoETS/BottomUp,AutoETS/MinTrace_method-mint_shrink,AutoETS/MinTrace_method-ols,y
0,Australia,2016-01-01,25990.068004,24381.672903,25427.793552,25894.419896,26660.637689
1,Australia,2016-04-01,24458.490282,22903.194015,23913.800914,24357.231461,24285.027757
2,Australia,2016-07-01,23974.055984,22411.401316,23428.540858,23865.928094,24191.320131
3,Australia,2016-10-01,24563.454495,23127.009693,24089.585592,24470.780870,26347.600973
4,Australia,2017-01-01,25990.068004,24518.047370,25545.039186,25901.383310,27496.389021


In [41]:
evaluation = evaluate(df = df,
                      tags = eval_tags,
                      train_df = Y_train_df,
                      metrics = [rmse,
                                 partial(mase, seasonality=4)])

evaluation.columns = ['level', 'metric', 'Base', 'BottomUp', 'MinTrace(mint_shrink)', 'MinTrace(ols)']
numeric_cols = evaluation.select_dtypes(include="number").columns
evaluation[numeric_cols] = evaluation[numeric_cols].map('{:.2f}'.format).astype(np.float64)

In [44]:
evaluation.query('metric == "mase"')


,level,metric,Base,BottomUp,MinTrace(mint_shrink),MinTrace(ols)
1,Total,mase,1.59,3.16,2.06,1.67
3,Purpose,mase,1.32,2.28,1.48,1.25
5,State,mase,1.39,1.90,1.40,1.25
7,Regions,mase,1.12,1.19,1.01,0.99
9,Bottom,mase,0.98,0.98,0.94,1.01
11,Overall,mase,1.02,1.06,0.97,1.02


In [43]:
evaluation.query('metric == "rmse"')

,level,metric,Base,BottomUp,MinTrace(mint_shrink),MinTrace(ols)
0,Total,rmse,1743.29,3029.02,2112.94,1818.94
2,Purpose,rmse,534.75,791.28,577.18,515.53
4,State,rmse,308.15,413.44,316.85,287.34
6,Regions,rmse,51.66,55.13,46.55,46.29
8,Bottom,rmse,19.37,19.37,17.80,18.19
10,Overall,rmse,41.13,49.82,40.47,38.75
